# DermFuse (Kaggle): resumable experiments, imbalance screening, bootstrap CIs, grouped SHAP, TTA

HAM10000, 7 classes. CNN (EfficientNet-B0) vs Transformer (DeiT-Small), each image-only and fused with age / sex / body site.

## How to run on Kaggle
1. **File > Import Notebook**, upload this file. In the right-hand panel set **Accelerator = GPU T4 x2** (if you get `no kernel image is available`, you are on a P100 with a newer PyTorch: switch to T4) and **Internet = On** (timm downloads the pretrained weights).
2. **Add Input** and search for **Skin Cancer MNIST: HAM10000** (by Kmader). That is all the data setup.
3. **Before every commit, check the right-hand panel:** *Session options* shows the GPU accelerator and *Input* lists the HAM10000 dataset. The notebook stops with a clear message if either is missing. Pick a `STAGE` in the config cell and press **Save Version > Save & Run All (Commit)**. Commits run in the background, survive closing the browser, and only count GPU time while running.
4. To carry results to the next session: open the finished version, **Output** tab, create a private **Dataset** from the files (or update the one you made), and **Add Input** it to the next session. The notebook finds any attached `dermfuse_out` folder (split file and finished runs) by itself, skips runs that are already complete and have the same configuration, and keeps the same train/val/test split.

## Stages (run in this order, one per session)
| STAGE | What it does | GPU time (T4, approx.) |
|---|---|---|
| `screen` | CNN-fused, seed 42, three imbalance strategies (baseline, power 1.0, focal) | about 45 min |
| `analysis` | Reads saved outputs: screening decision (validation only), logit adjustment, mean/std, lesion-level bootstrap CIs, paired comparisons | CPU, minutes |
| `final` | Four image configs + metadata-only MLP for the seeds in `SEEDS_THIS_RUN` under `STRATEGY_FOR_FINAL` | about 1 h per seed |
| `xai` | Test-time augmentation, metadata-shuffle test, grouped Shapley attribution, Grad-CAM / attention rollout | about 5-10 min |

Run `screen`, then `analysis` (read the DECISION lines), set `STRATEGY_FOR_FINAL`, then run `final` three times with `SEEDS_THIS_RUN = [42]`, `[1]`, `[2]`, then `analysis` and `xai`.

## Rules fixed in advance (so nothing is tuned on the test set)
- Model selection and strategy selection use **validation** data only. The screening report shows validation columns only.
- A new imbalance strategy replaces the baseline only if its validation macro-F1 is at least `SELECTION_DELTA` higher **and** validation melanoma sensitivity does not drop.
- Logit adjustment: one procedure for every model. Tau is chosen on that model's validation logits from a coarse grid, using a smoothed validation curve, and kept at 0 unless the gain is at least `SELECTION_DELTA`.
- Primary metric: macro-F1. Primary contrasts: CNN vs DeiT, and fused vs image-only. Melanoma sensitivity is secondary.
- Bootstrap CIs resample **lesions** (all images of a sampled lesion stay together). Seed standard deviation and bootstrap CIs describe different sources of variation; report both and keep them separate.

## Two GPUs (T4 x2)
For stages `screen` and `final` the last cell starts two worker processes, one per GPU, each taking half of the jobs (`final`: CNN jobs on one GPU, ViT jobs on the other). Progress lines appear in the cell prefixed `[w0]` / `[w1]`. Use **Save & Run All (Commit)**, or restart the session and Run All; the worker script is rebuilt from the executed cells, so re-running cells by hand in an interactive session can break it.

In [ ]:
import importlib, subprocess, sys
for pkg in ["timm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", pkg], check=True)

In [ ]:
import os, sys, math, time, json, random, hashlib, shutil, platform, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import timm, sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
try:
    from IPython.display import display
except Exception:
    display = print
warnings.filterwarnings("ignore")

# multi-GPU sharding: set by the launcher in the last cell for its worker processes (leave as is)
SHARD = int(os.environ.get("DERMFUSE_SHARD", 0))
N_SHARDS = int(os.environ.get("DERMFUSE_N_SHARDS", 1))

# ======================= WHAT TO RUN =======================
STAGE = "screen"              # screen | analysis | final | xai
SEEDS_THIS_RUN = [42]         # stage "final": run [42], then [1], then [2], each in its own session
STRATEGY_FOR_FINAL = "ce_p0.5"  # set after reading the DECISION line of stage "analysis"
SCREEN_BACKBONES = ["cnn"]    # add "vit" to also screen on DeiT-Small
APPLY_LA = False              # also report logit-adjusted variants in the final tables
DRY_RUN = False               # True: ~700 images, 2 epochs, separate output folder (pipeline check)
# ===========================================================

class CFG:
    img_size = 224
    classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
    split_seed = 42
    n_folds = 20                  # 3 folds test + 3 folds val + 14 folds train = 15 / 15 / 70
    epochs = 25
    patience = 6                  # early stopping on validation macro-F1
    batch_size = 32
    num_workers = 2
    weight_decay = 1e-2
    head_lr = 1e-3
    backbone_lr = {"cnn": 1e-4, "vit": 5e-5}
    warmup_epochs = {"cnn": 1, "vit": 2}
    focal_gamma = 2.0
    pretrained = True
    backbones = {"cnn": "efficientnet_b0", "vit": "deit_small_patch16_224"}
    amp = True

# strategy -> loss and class-weight power (baseline = the setting used so far)
STRATEGIES = {
    "ce_p0.5": dict(loss="ce", power=0.5),      # baseline: class-weighted cross-entropy
    "ce_p1.0": dict(loss="ce", power=1.0),      # full inverse-frequency weights
    "focal":   dict(loss="focal", power=0.5),   # focal loss (gamma 2) with the baseline weights
}
BASELINE = "ce_p0.5"
SELECTION_DELTA = 0.02        # minimum validation macro-F1 gain to replace the baseline / switch on logit adjustment
TAU_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
N_BOOT = 2000

if DRY_RUN:
    CFG.epochs = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALLOW_CPU = False             # training on CPU would take many hours; leave False
if STAGE in ("screen", "final", "xai") and device.type != "cuda" and not (ALLOW_CPU or DRY_RUN):
    raise RuntimeError("No GPU found. In the right-hand panel open Session options and set Accelerator to GPU T4 x2, "
                       "then Save Version again (a commit uses the accelerator that is set when you save).")
S = CFG.img_size
NC = len(CFG.classes)
MEL = CFG.classes.index("mel")
ENV = dict(torch=torch.__version__, timm=timm.__version__, sklearn=sklearn.__version__,
           python=platform.python_version(), cuda=torch.version.cuda,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print("stage:", STAGE, "| device:", device, "|", ENV["gpu"])

## 1. Paths, restore earlier results, data, fixed split

In [ ]:
ON_KAGGLE = Path("/kaggle/working").exists()
BASE_OUT = Path("/kaggle/working") if ON_KAGGLE else Path("/content" if Path("/content").exists() else ".")
OUT = BASE_OUT / ("dermfuse_dry" if DRY_RUN else "dermfuse_out")
RUNS = OUT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
CACHE = Path("/tmp/dermfuse_cache"); CACHE.mkdir(parents=True, exist_ok=True)

def find_prior_roots():
    base = Path("/kaggle/input"); roots = set()
    if base.exists():
        for pat in ["*/split.csv", "*/*/split.csv", "*/*/*/split.csv"]:
            roots |= {p.parent for p in base.glob(pat)}
        for pat in ["*/runs", "*/*/runs", "*/*/*/runs"]:
            roots |= {p.parent for p in base.glob(pat)}
    return sorted(roots)

def restore_prior():
    copied = 0
    for root in find_prior_roots():
        if root.resolve() == OUT.resolve():
            continue
        if (root / "split.csv").exists() and not (OUT / "split.csv").exists():
            shutil.copy(root / "split.csv", OUT / "split.csv"); print("restored split from", root)
        if (root / "runs").exists():
            for d in (root / "runs").iterdir():
                if (d / "run.json").exists() and not (RUNS / d.name).exists():
                    shutil.copytree(d, RUNS / d.name); copied += 1
    print(f"restored {copied} finished run(s) from attached inputs")

if not DRY_RUN:
    restore_prior()

def find_ham():
    for base in ["/kaggle/input", "/content", "."]:
        if os.path.isdir(base):
            for root, dirs, files in os.walk(base, followlinks=True):
                if "HAM10000_metadata.csv" in files:
                    return Path(root)
    listing = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "(no /kaggle/input folder)"
    raise FileNotFoundError("HAM10000 not found. Contents of /kaggle/input: " + str(listing) +
                            ". In the right-hand panel use Input > Add Input, search 'Skin Cancer MNIST: HAM10000' "
                            "(by Kmader), check it is listed under Input, then Save Version again.")

DATA_DIR = find_ham()
df = pd.read_csv(DATA_DIR / "HAM10000_metadata.csv")
df["sex"] = df["sex"].fillna("unknown")
df["localization"] = df["localization"].fillna("unknown")
img_paths = {}
for root, dirs, files in os.walk(DATA_DIR, followlinks=True):
    for f in files:
        if f.lower().endswith(".jpg"):
            img_paths[os.path.splitext(f)[0]] = Path(root) / f
df["path"] = df["image_id"].map(img_paths)
assert df["path"].notna().all(), "some images are missing"
df["label"] = df["dx"].map({c: i for i, c in enumerate(CFG.classes)})
if DRY_RUN:
    df = df.sample(frac=1, random_state=0).groupby("dx").head(100).reset_index(drop=True)
print(len(df), "images,", df["lesion_id"].nunique(), "lesions | data:", DATA_DIR)

# ---- fixed split: created once, then always loaded from split.csv ----
split_file = OUT / "split.csv"
if split_file.exists():
    sp = pd.read_csv(split_file)
    df = df.merge(sp[["image_id", "split", "fold"]], on="image_id", how="left")
    assert df["split"].notna().all(), "split.csv does not cover this dataset"
    print("loaded saved split")
else:
    sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.split_seed)
    df["fold"] = -1
    for k, (_, ix) in enumerate(sgkf.split(df, df["label"], groups=df["lesion_id"])):
        df.loc[df.index[ix], "fold"] = k
    df["split"] = np.where(df.fold < 3, "test", np.where(df.fold < 6, "val", "train"))
    df[["image_id", "lesion_id", "dx", "split", "fold"]].to_csv(split_file, index=False)
    print("created and saved split")
SPLIT_HASH = hashlib.md5(split_file.read_bytes()).hexdigest()[:8]

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not set(df[df.split == a].lesion_id) & set(df[df.split == b].lesion_id), "lesion leakage"
tab = pd.crosstab(df["dx"], df["split"]).reindex(CFG.classes)[["train", "val", "test"]]
tab.loc["total"] = tab.sum()
display(tab)
print("split hash:", SPLIT_HASH)

## 2. Metadata, images, loaders

In [ ]:
SEX = ["female", "male", "unknown"]
SITES = sorted(df["localization"].unique())
tr = df[df.split == "train"]
age_med = float(tr["age"].median())
age_filled = tr["age"].fillna(age_med)
age_mu, age_sd = float(age_filled.mean()), float(age_filled.std() + 1e-8)
META_COLS = ["age_z", "age_missing"] + [f"sex_{s}" for s in SEX] + [f"site_{s}" for s in SITES]

def encode_meta(d):
    X = np.zeros((len(d), len(META_COLS)), np.float32)
    X[:, 0] = ((d["age"].fillna(age_med) - age_mu) / age_sd).values
    X[:, 1] = d["age"].isna().values
    for j, s in enumerate(SEX):
        X[:, 2 + j] = (d["sex"] == s).values
    for j, s in enumerate(SITES):
        X[:, 2 + len(SEX) + j] = (d["localization"] == s).values
    return X

X_all = encode_meta(df)
y_all = df["label"].values.astype(np.int64)
META_DIM = X_all.shape[1]
idx = {s: np.where(df.split.values == s)[0] for s in ["train", "val", "test"]}

NEED_IMAGES = STAGE in ("screen", "final", "xai")
if NEED_IMAGES:
    key = hashlib.md5("".join(df["image_id"]).encode()).hexdigest()[:8]
    cache = CACHE / f"imgs_{S}_{len(df)}_{key}.npy"
    def load_resized(p):
        return np.asarray(Image.open(p).convert("RGB").resize((S, S), Image.BILINEAR), dtype=np.uint8)
    if cache.exists():
        IMGS = np.load(cache)
    else:
        with ThreadPoolExecutor(8) as ex:
            IMGS = np.stack(list(tqdm(ex.map(load_resized, df["path"].tolist()), total=len(df))))
        np.save(cache, IMGS)
    print("images:", IMGS.shape)

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomResizedCrop(S, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(180),
    T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(), T.Normalize(mean, std)])
eval_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

class LesionDS(Dataset):
    def __init__(self, ids, X, train=False, use_images=True):
        self.ids, self.X, self.tf, self.use_images = ids, X, (train_tf if train else eval_tf), use_images
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        img = self.tf(Image.fromarray(IMGS[j])) if self.use_images else torch.zeros(1)
        return img, torch.from_numpy(self.X[j]), int(y_all[j])

def make_loader(split, train=False, use_images=True, X=None, ids=None):
    ds = LesionDS(idx[split] if ids is None else ids, X_all if X is None else X, train, use_images)
    return DataLoader(ds, batch_size=CFG.batch_size, shuffle=train, drop_last=train,
                      num_workers=CFG.num_workers, pin_memory=device.type == "cuda")

## 3. Model

In [ ]:
class MetaMLP(nn.Module):
    def __init__(self, d_in, d_hidden=64, d_out=32, p=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(d_hidden, d_out), nn.ReLU())
        self.out_dim = d_out
    def forward(self, x): return self.net(x)

class FusionNet(nn.Module):
    def __init__(self, backbone_key, mode, meta_dim, n_classes=NC):
        super().__init__()
        assert mode in ("image", "meta", "fused")
        self.mode, self.backbone, self.meta = mode, None, None
        d = 0
        if mode in ("image", "fused"):
            kw = {"img_size": S} if backbone_key == "vit" else {}
            self.backbone = timm.create_model(CFG.backbones[backbone_key], pretrained=CFG.pretrained,
                                              num_classes=0, **kw)
            d += self.backbone.num_features
        if mode in ("meta", "fused"):
            self.meta = MetaMLP(meta_dim); d += self.meta.out_dim
        self.head = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_classes))
    def features(self, img, meta):
        f = []
        if self.backbone is not None: f.append(self.backbone(img))
        if self.meta is not None: f.append(self.meta(meta))
        return torch.cat(f, 1)
    def forward(self, img, meta):
        return self.head(self.features(img, meta))

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__(); self.weight, self.gamma = weight, gamma
    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, weight=self.weight, reduction="none")
        pt = torch.exp(-F.cross_entropy(logits, y, reduction="none"))
        return ((1 - pt) ** self.gamma * ce).mean()

def class_weights(power):
    c = np.bincount(y_all[idx["train"]], minlength=NC).astype(np.float64)
    w = (c.sum() / (NC * np.maximum(c, 1))) ** power
    w = w / (w * c).sum() * c.sum()
    return torch.tensor(w, dtype=torch.float32)

def make_criterion(strategy):
    s = STRATEGIES[strategy]
    w = class_weights(s["power"]).to(device)
    return FocalLoss(w, CFG.focal_gamma) if s["loss"] == "focal" else nn.CrossEntropyLoss(weight=w)

## 4. Metrics and analysis utilities (numpy only, no GPU needed)

In [ ]:
def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def metrics_from_cm(cms):
    # cms: (K, K) or (B, K, K); rows = true class. Works with weighted (bootstrap) confusion matrices.
    cms = np.asarray(cms, dtype=np.float64)
    if cms.ndim == 2:
        cms = cms[None]
    tp = np.diagonal(cms, axis1=1, axis2=2)
    rs, cs, tot = cms.sum(2), cms.sum(1), cms.sum((1, 2))
    sens = tp / np.maximum(rs, 1e-12)
    spec = (tot[:, None] - rs - cs + tp) / np.maximum(tot[:, None] - rs, 1e-12)
    f1 = 2 * tp / np.maximum(rs + cs, 1e-12)
    present = rs > 0
    return dict(acc=tp.sum(1) / tot,
                bal_acc=(sens * present).sum(1) / np.maximum(present.sum(1), 1),
                macro_f1=f1.mean(1), mel_sens=sens[:, MEL], sens=sens, spec=spec, f1=f1)

def cm_from_preds(y, pred):
    return np.bincount(np.asarray(y) * NC + np.asarray(pred), minlength=NC * NC).reshape(NC, NC)

def compute_metrics(y_true, logits):
    logits = np.asarray(logits)
    cm = cm_from_preds(y_true, logits.argmax(1))
    m = metrics_from_cm(cm)
    try:
        auc = roc_auc_score(y_true, softmax_np(logits), multi_class="ovr", average="macro", labels=list(range(NC)))
    except ValueError:
        auc = float("nan")
    return dict(acc=float(m["acc"][0]), bal_acc=float(m["bal_acc"][0]), macro_f1=float(m["macro_f1"][0]),
                auc=float(auc), mel_sens=float(m["mel_sens"][0]), sens=m["sens"][0], spec=m["spec"][0], cm=cm)

def metrics_row(m):
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in m.items() if k != "cm"}

# ---------- lesion-level bootstrap ----------
def lesion_boot_weights(lesion_codes, B, seed=0):
    rng = np.random.default_rng(seed)
    L = int(lesion_codes.max()) + 1
    counts = rng.multinomial(L, np.full(L, 1.0 / L), size=B)          # (B, L): how often each lesion is drawn
    return counts[:, lesion_codes].astype(np.float64)                 # (B, n_images)

def boot_cms(y, pred, W):
    n = len(y)
    M = np.zeros((n, NC * NC)); M[np.arange(n), np.asarray(y) * NC + np.asarray(pred)] = 1.0
    return (W @ M).reshape(len(W), NC, NC)

def ci(x, q=(2.5, 97.5)):
    return tuple(np.percentile(x, q))

# ---------- logit adjustment ----------
def tune_tau(val_logits, yv, prior, grid=None, delta=None):
    grid = TAU_GRID if grid is None else grid
    delta = SELECTION_DELTA if delta is None else delta
    f1s = np.array([metrics_from_cm(cm_from_preds(yv, (val_logits - t * np.log(prior)).argmax(1)))["macro_f1"][0]
                    for t in grid])
    sm = np.convolve(np.pad(f1s, (1, 1), mode="edge"), np.ones(3) / 3, mode="valid")   # prefer stable regions
    best = int(sm.argmax())
    if f1s[best] - f1s[0] < delta:
        best = 0                                                                        # no adjustment unless clearly better
    return grid[best], f1s

# ---------- run bookkeeping ----------
def load_all_runs():
    rows = []
    for d in sorted(RUNS.iterdir()):
        j = d / "run.json"
        if j.exists():
            info = json.load(open(j))
            if info.get("status") == "complete":
                info["dir"] = str(d); rows.append(info)
    return rows

## 5. Training with saved outputs (resumable at run level)

In [ ]:
def run_id(bk, mode, strat, seed):
    return f"{bk or 'none'}_{mode}_{strat}_s{seed}"

def cfg_signature(bk, mode, strat):
    d = dict(bk=bk, mode=mode, strat=strat, strat_cfg=STRATEGIES[strat], img=S, epochs=CFG.epochs,
             patience=CFG.patience, bs=CFG.batch_size, wd=CFG.weight_decay, head_lr=CFG.head_lr,
             blr=CFG.backbone_lr.get(bk), warm=CFG.warmup_epochs.get(bk), gamma=CFG.focal_gamma,
             model=CFG.backbones.get(bk), pretrained=CFG.pretrained, split=SPLIT_HASH)
    return hashlib.md5(json.dumps(d, sort_keys=True).encode()).hexdigest()[:10]

def is_complete(bk, mode, strat, seed):
    j = RUNS / run_id(bk, mode, strat, seed) / "run.json"
    if not j.exists():
        return False
    info = json.load(open(j))
    return info.get("status") == "complete" and info.get("sig") == cfg_signature(bk, mode, strat)

@torch.no_grad()
def predict(model, loader):
    model.eval(); out, ys = [], []
    for img, meta, y in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=CFG.amp and device.type == "cuda"):
            out.append(model(img.to(device), meta.to(device)).float().cpu())
        ys.append(y)
    return torch.cat(out).numpy(), torch.cat(ys).numpy()

def fit_and_save(bk, mode, strat, seed, print_test=True):
    rid = run_id(bk, mode, strat, seed)
    set_seed(seed)
    use_images = mode != "meta"
    tl, vl = make_loader("train", True, use_images), make_loader("val", False, use_images)
    model = FusionNet(bk, mode, META_DIM).to(device)

    blr = CFG.backbone_lr.get(bk, CFG.head_lr)
    groups = [{"params": [p for n, p in model.named_parameters() if n.startswith("backbone")], "lr": blr},
              {"params": [p for n, p in model.named_parameters() if not n.startswith("backbone")], "lr": CFG.head_lr}]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)
    total = CFG.epochs * len(tl)
    warm = CFG.warmup_epochs.get(bk, 1) * len(tl) if use_images else 0
    lam = lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lam)
    use_amp = CFG.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    crit = make_criterion(strat)

    best, best_state, bad, hist, t_ep = -1, None, 0, [], []
    for ep in range(CFG.epochs):
        t0 = time.time(); model.train(); run = 0.0; n = 0
        for img, meta, y in tl:
            img, meta, y = img.to(device, non_blocking=True), meta.to(device), y.to(device)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = crit(model(img, meta).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * len(y); n += len(y)
        lg, yt = predict(model, vl); m = compute_metrics(yt, lg)
        t_ep.append(time.time() - t0)
        hist.append(dict(epoch=ep + 1, train_loss=run / n, val_f1=m["macro_f1"], val_bal_acc=m["bal_acc"]))
        print(f"  ep {ep+1:02d} | loss {run/n:.3f} | val macro-F1 {m['macro_f1']:.3f} | val bal-acc {m['bal_acc']:.3f} | {t_ep[-1]:.0f}s")
        if m["macro_f1"] > best:
            best, bad = m["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG.patience: break

    model.load_state_dict(best_state)
    vlg, vy = predict(model, vl)
    tlg, ty = predict(model, make_loader("test", False, use_images))
    rd = RUNS / rid; rd.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, rd / "best.pt")
    np.save(rd / "val_logits.npy", vlg); np.save(rd / "val_y.npy", vy)
    np.save(rd / "test_logits.npy", tlg); np.save(rd / "test_y.npy", ty)
    pd.DataFrame(hist).to_csv(rd / "history.csv", index=False)
    info = dict(run_id=rid, status="complete", sig=cfg_signature(bk, mode, strat), backbone=bk, mode=mode,
                strategy=strat, seed=seed, best_val_f1=best, epochs_run=len(hist),
                params_m=sum(p.numel() for p in model.parameters()) / 1e6, sec_per_epoch=float(np.mean(t_ep)),
                val=metrics_row(compute_metrics(vy, vlg)), test=metrics_row(compute_metrics(ty, tlg)),
                env=ENV, config=dict(epochs=CFG.epochs, patience=CFG.patience, batch=CFG.batch_size,
                                     split=SPLIT_HASH, strategy=STRATEGIES[strat]))
    json.dump(info, open(rd / "run.json", "w"), indent=1)          # written last = marks the run as complete
    msg = f"  saved {rid} | val macro-F1 {best:.3f}"
    if print_test:
        msg += f" | TEST macro-F1 {info['test']['macro_f1']:.3f} bal-acc {info['test']['bal_acc']:.3f} mel-sens {info['test']['mel_sens']:.3f}"
    print(msg)
    del model, opt, tl, vl
    import gc; gc.collect(); torch.cuda.empty_cache()

def job_list_all():
    if STAGE == "screen":
        return [(bk, "fused", s, 42) for bk in SCREEN_BACKBONES for s in STRATEGIES]
    jobs = []
    for seed in SEEDS_THIS_RUN:
        jobs += [(bk, mode, STRATEGY_FOR_FINAL, seed) for bk, mode in
                 [("cnn", "image"), ("vit", "image"), ("cnn", "fused"), ("vit", "fused")]]
        jobs.append((None, "meta", STRATEGY_FOR_FINAL, seed))
    return jobs

def job_list():
    return job_list_all()[SHARD::N_SHARDS]

def run_training_stage():
    jobs = job_list()
    print(f"{len(jobs)} job(s):", [run_id(*j) for j in jobs])
    for bk, mode, strat, seed in jobs:
        rid = run_id(bk, mode, strat, seed)
        if is_complete(bk, mode, strat, seed):
            print("skip (complete, same configuration):", rid); continue
        print("\n===", rid, "===")
        fit_and_save(bk, mode, strat, seed, print_test=(STAGE != "screen"))

## 6. Analysis stage (CPU is enough)

In [ ]:
def _load_logits(r, split, variant, prior):
    lg = np.load(Path(r["dir"]) / f"{split}_logits.npy").astype(np.float64)
    if variant == "la":
        tau, _ = tune_tau(np.load(Path(r["dir"]) / "val_logits.npy").astype(np.float64),
                          np.load(Path(r["dir"]) / "val_y.npy"), prior)
        lg = lg - tau * np.log(prior)
    return lg

def run_analysis():
    A = OUT / "analysis"; A.mkdir(exist_ok=True)
    sp = pd.read_csv(OUT / "split.csv")
    cls_idx = {c: i for i, c in enumerate(CFG.classes)}
    prior = sp.loc[sp.split == "train", "dx"].value_counts().reindex(CFG.classes).fillna(0).values.astype(float)
    prior = prior / prior.sum()
    test_sp = sp[sp.split == "test"]
    test_les = pd.factorize(test_sp["lesion_id"])[0]
    y_check = test_sp["dx"].map(cls_idx).values
    runs = load_all_runs()
    print(len(runs), "complete run(s) found")
    for r in runs:                                   # alignment guard: saved labels must match split.csv order
        assert (np.load(Path(r["dir"]) / "test_y.npy") == y_check).all(), f"label mismatch in {r['run_id']}"

    # ---- 1. screening (validation only) ----
    scr = [r for r in runs if r["mode"] == "fused" and r["seed"] == 42 and r["strategy"] in STRATEGIES]
    if scr:
        rows = [dict(backbone=r["backbone"], strategy=r["strategy"], val_macro_f1=r["val"]["macro_f1"],
                     val_bal_acc=r["val"]["bal_acc"], val_mel_sens=r["val"]["mel_sens"],
                     **{f"val_rec_{c}": r["val"]["sens"][i] for i, c in enumerate(CFG.classes)}) for r in scr]
        sc = pd.DataFrame(rows).sort_values(["backbone", "strategy"])
        sc.to_csv(A / "screening_val.csv", index=False)
        print("\nSCREENING (validation metrics only; test is not used for this decision)")
        display(sc.round(3))
        for bk in sc["backbone"].unique():
            sub = sc[sc.backbone == bk].set_index("strategy")
            if BASELINE not in sub.index:
                print(f"DECISION ({bk}): baseline run missing, cannot decide"); continue
            base = sub.loc[BASELINE]
            ok = [s for s in sub.index if s != BASELINE
                  and sub.loc[s, "val_macro_f1"] >= base["val_macro_f1"] + SELECTION_DELTA
                  and sub.loc[s, "val_mel_sens"] >= base["val_mel_sens"]]
            pick = max(ok, key=lambda s: sub.loc[s, "val_macro_f1"]) if ok else BASELINE
            print(f"DECISION ({bk}): use '{pick}'" + ("" if ok else f"  (no strategy beat the baseline by >= {SELECTION_DELTA} with melanoma sensitivity preserved)"))

    # ---- 2. logit adjustment per run (tau tuned on validation) ----
    la_rows = []
    for r in runs:
        vl = np.load(Path(r["dir"]) / "val_logits.npy").astype(np.float64); vy = np.load(Path(r["dir"]) / "val_y.npy")
        tl = np.load(Path(r["dir"]) / "test_logits.npy").astype(np.float64); ty = np.load(Path(r["dir"]) / "test_y.npy")
        tau, f1s = tune_tau(vl, vy, prior)
        m0, m1 = compute_metrics(ty, tl), compute_metrics(ty, tl - tau * np.log(prior))
        la_rows.append(dict(run_id=r["run_id"], tau=tau, val_f1_tau0=f1s[0], val_f1_at_tau=f1s[TAU_GRID.index(tau)],
                            test_f1_raw=m0["macro_f1"], test_f1_la=m1["macro_f1"], test_mel_raw=m0["mel_sens"],
                            test_mel_la=m1["mel_sens"], test_bal_raw=m0["bal_acc"], test_bal_la=m1["bal_acc"]))
    if la_rows:
        la = pd.DataFrame(la_rows); la.to_csv(A / "logit_adjustment.csv", index=False)
        print("\nLOGIT ADJUSTMENT (tau chosen on validation; tau=0 means no adjustment was justified)")
        display(la.round(3))

    # ---- 3. final tables for the chosen strategy ----
    final = [r for r in runs if r["strategy"] == STRATEGY_FOR_FINAL]
    cfgs = {}
    for r in final:
        cfgs.setdefault(f"{r['backbone'] or 'mlp'}_{r['mode']}", []).append(r)
    if not cfgs:
        print("\nNo runs for STRATEGY_FOR_FINAL yet."); return

    summ = []
    for name, rs in cfgs.items():
        row = dict(config=name, n_seeds=len(rs), seeds=",".join(str(r["seed"]) for r in sorted(rs, key=lambda r: r["seed"])))
        for met in ["macro_f1", "bal_acc", "acc", "auc", "mel_sens"]:
            v = np.array([r["test"][met] for r in rs])
            row[met + "_mean"] = v.mean(); row[met + "_sd"] = v.std(ddof=1) if len(v) > 1 else np.nan
        summ.append(row)
    summ = pd.DataFrame(summ); summ.to_csv(A / "summary_by_config.csv", index=False)
    print(f"\nTEST RESULTS, strategy '{STRATEGY_FOR_FINAL}': mean and SD across seeds (SD is a rough training-variability estimate)")
    display(summ.round(3))

    variants = ["raw"] + (["la"] if APPLY_LA else [])
    W = lesion_boot_weights(test_les, N_BOOT, seed=0)
    ytest = y_check
    agg, ci_rows, cls_rows = {}, [], []
    for v in variants:
        for name, rs in cfgs.items():
            per_b, per_pt = [], []
            for r in rs:
                lg = _load_logits(r, "test", v, prior)
                per_b.append(metrics_from_cm(boot_cms(ytest, lg.argmax(1), W)))
                per_pt.append(compute_metrics(ytest, lg))
            a = {m: np.mean([b[m] for b in per_b], axis=0) for m in ["macro_f1", "bal_acc", "acc", "mel_sens", "sens"]}
            pt = {m: np.mean([p[m] for p in per_pt], axis=0) for m in ["macro_f1", "bal_acc", "acc", "mel_sens", "sens"]}
            agg[(v, name)] = (a, pt)
            for m in ["macro_f1", "bal_acc", "acc", "mel_sens"]:
                lo, hi = ci(a[m]); ci_rows.append(dict(variant=v, config=name, metric=m, estimate=pt[m], ci_lo=lo, ci_hi=hi, n_seeds=len(rs)))
            for k, c in enumerate(CFG.classes):
                lo, hi = ci(a["sens"][:, k])
                cls_rows.append(dict(variant=v, config=name, cls=c, recall=pt["sens"][k], ci_lo=lo, ci_hi=hi,
                                     test_images=int((ytest == k).sum()), test_lesions=int(test_sp.loc[ytest == k, "lesion_id"].nunique())))
    cit = pd.DataFrame(ci_rows); cit.to_csv(A / "bootstrap_ci.csv", index=False)
    clt = pd.DataFrame(cls_rows); clt.to_csv(A / "per_class_recall_ci.csv", index=False)
    print(f"\nLESION-LEVEL BOOTSTRAP 95% CIs ({N_BOOT} resamples; estimate = mean over seeds)")
    display(cit.round(3))
    print("\nPER-CLASS RECALL with CIs (rare classes have very few test lesions, so intervals are wide)")
    display(clt.round(3))

    pairs = [("vit_image", "cnn_image"), ("vit_fused", "cnn_fused"), ("cnn_fused", "cnn_image"), ("vit_fused", "vit_image")]
    prow = []
    for v in variants:
        for a_name, b_name in pairs:
            if (v, a_name) in agg and (v, b_name) in agg:
                (aa, ap), (ba, bp) = agg[(v, a_name)], agg[(v, b_name)]
                for m in ["macro_f1", "bal_acc", "mel_sens"]:
                    d = aa[m] - ba[m]; lo, hi = ci(d)
                    prow.append(dict(variant=v, contrast=f"{a_name} - {b_name}", metric=m, difference=ap[m] - bp[m],
                                     ci_lo=lo, ci_hi=hi, excludes_zero=bool(lo > 0 or hi < 0)))
    if prow:
        pt_ = pd.DataFrame(prow); pt_.to_csv(A / "paired_differences.csv", index=False)
        print("\nPAIRED LESION-LEVEL BOOTSTRAP DIFFERENCES")
        display(pt_.round(3))
    print("\nsaved tables to", A)

## 7. XAI stage: test-time augmentation, metadata shuffle, grouped Shapley, Grad-CAM / attention rollout

In [ ]:
def load_model(rid):
    info = json.load(open(RUNS / rid / "run.json"))
    old = CFG.pretrained; CFG.pretrained = False
    m = FusionNet(info["backbone"], info["mode"], META_DIM).to(device)
    CFG.pretrained = old
    m.load_state_dict(torch.load(RUNS / rid / "best.pt", map_location=device)); m.eval()
    return m, info

def tta_views(x):
    out = []
    for k in range(4):
        r = torch.rot90(x, k, dims=(2, 3)); out += [r, torch.flip(r, dims=(3,))]
    return out                                                     # 8 fixed views: 4 rotations x optional flip

@torch.no_grad()
def predict_probs(model, loader, tta=False):
    model.eval(); P = []
    for img, meta, y in loader:
        img, meta = img.to(device), meta.to(device)
        views = tta_views(img) if tta else [img]
        acc = 0
        for v in views:
            with torch.autocast(device.type, dtype=torch.float16, enabled=CFG.amp and device.type == "cuda"):
                acc = acc + torch.softmax(model(v, meta).float(), 1)
        P.append((acc / len(views)).cpu())
    return torch.cat(P).numpy()

def grouped_shapley(f, x, B, groups):
    # exact Shapley values for a few feature GROUPS (interventional: absent groups come from background rows).
    names = list(groups); m = len(names); vals = {}
    for mask in range(2 ** m):
        X = B.copy()
        for gi, name in enumerate(names):
            if (mask >> gi) & 1:
                X[:, groups[name]] = x[groups[name]]
        vals[mask] = float(np.mean(f(X)))
    phi = np.zeros(m)
    for gi in range(m):
        for mask in range(2 ** m):
            if (mask >> gi) & 1:
                continue
            s = bin(mask).count("1")
            w = math.factorial(s) * math.factorial(m - s - 1) / math.factorial(m)
            phi[gi] += w * (vals[mask | (1 << gi)] - vals[mask])
    return phi, vals

GROUPS = {"age": [0, 1], "sex": [2, 3, 4], "body site": list(range(5, META_DIM))}

@torch.no_grad()
def image_features(model, ids, bs=64):
    model.eval(); out = []
    for k in range(0, len(ids), bs):
        batch = torch.stack([eval_tf(Image.fromarray(IMGS[j])) for j in ids[k:k + bs]]).to(device)
        with torch.autocast(device.type, dtype=torch.float16, enabled=CFG.amp and device.type == "cuda"):
            out.append(model.backbone(batch).float())
    return torch.cat(out)

def shap_for_run(model, ids, bg):
    Z = image_features(model, ids); rows = []
    for k, j in enumerate(ids):
        z = Z[k:k + 1]
        def f_cls(M, cls, z=z):
            with torch.no_grad():
                m = model.meta(torch.as_tensor(M, dtype=torch.float32, device=device))
                return torch.softmax(model.head(torch.cat([z.expand(len(M), -1), m], 1)), 1)[:, cls].cpu().numpy()
        with torch.no_grad():
            p = torch.softmax(model.head(torch.cat([z, model.meta(torch.as_tensor(X_all[j][None], device=device))], 1)), 1)
        cls = int(p.argmax())
        phi, _ = grouped_shapley(lambda M: f_cls(M, cls), X_all[j], bg, GROUPS)
        rows.append(dict(index=int(j), pred=CFG.classes[cls], **dict(zip(GROUPS, phi))))
    return pd.DataFrame(rows)

def run_xai():
    A = OUT / "analysis"; A.mkdir(exist_ok=True)
    runs = [r for r in load_all_runs() if r["strategy"] == STRATEGY_FOR_FINAL and r["mode"] != "meta"]
    if not runs:
        print("no finished image runs for STRATEGY_FOR_FINAL"); return
    rng = np.random.RandomState(42)
    sample = np.sort(rng.choice(idx["test"], min(100, len(idx["test"])), replace=False))
    pd.DataFrame({"image_id": df["image_id"].values[sample]}).to_csv(A / "shap_sample_ids.csv", index=False)
    bg = X_all[idx["train"]][np.random.RandomState(0).choice(len(idx["train"]), 50, replace=False)]
    X_perm = X_all.copy()
    X_perm[idx["test"]] = X_all[idx["test"]][np.random.RandomState(0).permutation(len(idx["test"]))]

    tta_rows, shuf_rows, shap_rows = [], [], []
    for r in runs:
        model, info = load_model(r["run_id"]); rid = r["run_id"]; print("\n", rid)
        ty = y_all[idx["test"]]
        p0 = predict_probs(model, make_loader("test", False, True)); p1 = predict_probs(model, make_loader("test", False, True), tta=True)
        m0, m1 = compute_metrics(ty, np.log(p0 + 1e-12)), compute_metrics(ty, np.log(p1 + 1e-12))
        row = dict(run_id=rid)
        for k in ["macro_f1", "bal_acc", "acc", "auc", "mel_sens"]:
            row[k + "_plain"], row[k + "_tta"], row[k + "_change"] = m0[k], m1[k], m1[k] - m0[k]
        for i, c in enumerate(CFG.classes):
            row[f"recall_{c}_change"] = m1["sens"][i] - m0["sens"][i]
        tta_rows.append(row)
        if info["mode"] == "fused":
            ps = predict_probs(model, make_loader("test", False, True, X=X_perm))
            ms = compute_metrics(ty, np.log(ps + 1e-12))
            shuf_rows.append(dict(run_id=rid, macro_f1=m0["macro_f1"], macro_f1_metadata_shuffled=ms["macro_f1"],
                                  drop=m0["macro_f1"] - ms["macro_f1"]))
            sh = shap_for_run(model, sample, bg); sh.insert(0, "run_id", rid); shap_rows.append(sh)
        del model; torch.cuda.empty_cache()

    tta = pd.DataFrame(tta_rows); tta.to_csv(A / "tta_results.csv", index=False)
    print("\nTEST-TIME AUGMENTATION (8 fixed views; all changes shown, including negative ones)")
    display(tta[["run_id", "macro_f1_plain", "macro_f1_tta", "macro_f1_change", "bal_acc_change", "mel_sens_change", "auc_change"]].round(4))
    if shuf_rows:
        shuf = pd.DataFrame(shuf_rows); shuf.to_csv(A / "metadata_shuffle.csv", index=False)
        print("\nMETADATA SHUFFLE TEST (drop in macro-F1 when metadata is permuted across test patients)")
        display(shuf.round(3))
        allsh = pd.concat(shap_rows); allsh.to_csv(A / "shapley_group_attribution_samples.csv", index=False)
        out = []
        rb = np.random.default_rng(0)
        for rid, g in allsh.groupby("run_id"):
            for name in GROUPS:
                v = np.abs(g[name].values)
                boots = [v[rb.integers(0, len(v), len(v))].mean() for _ in range(N_BOOT)]
                lo, hi = ci(boots)
                out.append(dict(run_id=rid, group=name, mean_abs_attribution=v.mean(), ci_lo=lo, ci_hi=hi, mean_signed=g[name].mean()))
        so = pd.DataFrame(out); so.to_csv(A / "shapley_group_summary.csv", index=False)
        print("\nGROUPED SHAPLEY ATTRIBUTION of the metadata to the predicted-class probability, given each lesion's image features")
        print("(model behaviour, not medical causation; CIs resample the sampled images; sample ids are saved)")
        display(so.round(4))

    # ---- Grad-CAM (CNN) and attention rollout (ViT) for the first fused run of each backbone ----
    def denorm(img):
        return (img * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]).permute(1, 2, 0).clamp(0, 1).numpy()
    def _norm(x):
        x = x - x.min(); return x / (x.max() + 1e-8)
    def gradcam(model, img, meta):
        model.eval(); img, meta = img.unsqueeze(0).to(device), meta.unsqueeze(0).to(device)
        fmap = model.backbone.forward_features(img); fmap.retain_grad()
        feats = [model.backbone.forward_head(fmap)]
        if model.meta is not None: feats.append(model.meta(meta))
        logits = model.head(torch.cat(feats, 1)); t = int(logits.argmax(1))
        model.zero_grad(); logits[0, t].backward()
        w = fmap.grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * fmap).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(S, S), mode="bilinear", align_corners=False)[0, 0]
        return _norm(cam.detach().cpu().numpy()), t, torch.softmax(logits, 1)[0].detach().cpu().numpy()
    def rollout(model, img, meta):
        model.eval(); maps, handles, flags = [], [], []
        for blk in model.backbone.blocks:
            flags.append(blk.attn.fused_attn); blk.attn.fused_attn = False
            handles.append(blk.attn.attn_drop.register_forward_hook(lambda m_, i, o: maps.append(i[0].detach())))
        with torch.no_grad():
            logits = model(img.unsqueeze(0).to(device), meta.unsqueeze(0).to(device))
        for h in handles: h.remove()
        for blk, f in zip(model.backbone.blocks, flags): blk.attn.fused_attn = f
        Am = torch.stack(maps)[:, 0].mean(1); N = Am.shape[-1]; I = torch.eye(N, device=Am.device)
        Am = Am + I; Am = Am / Am.sum(-1, keepdim=True); R = I.clone()
        for a in Am: R = a @ R
        g = int(round((N - 1) ** 0.5))
        m_ = F.interpolate(R[0, 1:].reshape(1, 1, g, g), size=(S, S), mode="bilinear", align_corners=False)[0, 0]
        t = int(logits.argmax(1))
        return _norm(m_.cpu().numpy()), t, torch.softmax(logits, 1)[0].cpu().numpy()
    ex_rng = np.random.RandomState(0); examples = []
    for c in range(NC):
        ids = [i for i in idx["test"] if y_all[i] == c]
        if ids: examples.append(int(ex_rng.choice(ids)))
    for bk, fn, title in [("cnn", gradcam, "Grad-CAM (CNN)"), ("vit", rollout, "Attention rollout (ViT)")]:
        cand = [r for r in runs if r["backbone"] == bk and r["mode"] == "fused"]
        if not cand: continue
        model, _ = load_model(sorted(cand, key=lambda r: r["seed"])[0]["run_id"])
        fig, ax = plt.subplots(2, len(examples), figsize=(2.3 * len(examples), 5))
        for k, j in enumerate(examples):
            img = eval_tf(Image.fromarray(IMGS[j])); meta = torch.from_numpy(X_all[j])
            cam, t, prob = fn(model, img, meta)
            ax[0, k].imshow(denorm(img)); ax[0, k].axis("off"); ax[0, k].set_title(f"true {CFG.classes[y_all[j]]}", fontsize=9)
            ax[1, k].imshow(denorm(img)); ax[1, k].imshow(cam, cmap="jet", alpha=0.45); ax[1, k].axis("off")
            ax[1, k].set_title(f"pred {CFG.classes[t]} ({prob[t]:.2f})", fontsize=9)
        plt.suptitle(title); plt.tight_layout(); plt.savefig(A / f"{bk}_explanations.png", dpi=130); plt.show()
        del model; torch.cuda.empty_cache()

## 8. Dispatcher

In [ ]:
import subprocess, re

def _worker_src(src):
    if "run_cell_magic(" in src:                       # %%time etc.: skip the whole cell
        return ""
    out = []
    for line in src.splitlines():
        if "get_ipython(" in line or line.lstrip().startswith(("%", "!")):
            out.append(re.match(r"\s*", line).group() + "pass")
        else:
            out.append(line)
    return "\n".join(out)

if STAGE in ("screen", "final") and torch.cuda.device_count() >= 2 and "DERMFUSE_SHARD" not in os.environ:
    # Two GPUs: run two worker processes, one per GPU, each with half of the jobs.
    # The worker script is rebuilt from the executed cells, so use Save & Run All (or restart, then Run All).
    script = ("import os, sys\nfrom pathlib import Path\n\n"
              + "\n\n".join(_worker_src(c) for c in In[1:-1] if "worker.py" not in c)
              + "\nrun_training_stage()\n")
    compile(script, "worker.py", "exec")               # fail here, with a clear error, if the script is invalid
    Path("worker.py").write_text(script)
    procs = [subprocess.Popen([sys.executable, "-u", "worker.py"], stdout=open(f"worker{i}.log", "w"), stderr=subprocess.STDOUT,
                              env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i), "DERMFUSE_SHARD": str(i), "DERMFUSE_N_SHARDS": "2"})
             for i in range(2)]
    pos = [0, 0]
    def _flush():
        for i in range(2):
            txt = Path(f"worker{i}.log").read_text()
            new, pos[i] = txt[pos[i]:], len(txt)
            for line in new.splitlines():
                print(f"[w{i}] {line}")
    while any(p.poll() is None for p in procs):
        time.sleep(30)
        _flush()
    _flush()
    assert all(p.returncode == 0 for p in procs), "a worker failed, see the [w0]/[w1] lines above"
elif STAGE in ("screen", "final"):
    run_training_stage()
elif STAGE == "analysis":
    run_analysis()
elif STAGE == "xai":
    run_xai()
else:
    raise ValueError("STAGE must be one of: screen | analysis | final | xai")

total = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file()) / 1e6
print(f"\nOutput folder {OUT}: {total:.0f} MB. Save this version, then create a Dataset from the output to carry it to the next session.")